# 01 · Exploração dos Dados

**Autor:** Cristian Matias de Souza · **Projeto:** DashVendas

Objetivo: conhecer o dataset de vendas do e-commerce, conferir a qualidade dos dados e responder às perguntas da análise.

> Toda a lógica fica no pacote `src/dashvendas`. O notebook só **chama** essas funções e mostra os resultados — assim o mesmo código serve para o notebook, para o pipeline e para o dashboard.

## 1. Carregar os dados
Os dados originais ficam em `data/raw/ecom_sales.csv` e **nunca** são editados à mão.

In [1]:
from dashvendas.data import load_sales, save_processed, total_by_country

df = load_sales()
df.head()

,InvoiceNo,Description,OrderValue,Quantity,CustomerID,Country
0,549185,PACK OF 20 NAPKINS PANTRY DESIGN,10.20,12,18272.0,United Kingdom
1,576381,NATURAL SLATE HEART CHALKBOARD,35.40,12,12839.0,United Kingdom
2,551192,36 PENCILS TUBE SKULLS,20.00,16,16188.0,United Kingdom
3,573553,SET 6 SCHOOL MILK BOTTLES IN CRATE,7.46,1,NaN,United Kingdom
4,539436,FINE WICKER HEART,2.51,1,NaN,United Kingdom


## 2. Qualidade dos dados
Tipos das colunas, valores nulos e duplicados.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    500 non-null    int64  
 1   Description  500 non-null    str    
 2   OrderValue   500 non-null    float64
 3   Quantity     500 non-null    int64  
 4   CustomerID   399 non-null    float64
 5   Country      500 non-null    str    
dtypes: float64(2), int64(2), str(2)
memory usage: 23.6 KB


In [3]:
print('Nulos por coluna:')
print(df.isna().sum())
print('\nLinhas duplicadas:', df.duplicated().sum())

Nulos por coluna:
InvoiceNo        0
Description      0
OrderValue       0
Quantity         0
CustomerID     101
Country          0
dtype: int64

Linhas duplicadas: 0


In [4]:
df.describe()

,InvoiceNo,OrderValue,Quantity,CustomerID
count,500.000000,500.000000,500.000000,399.000000
mean,558862.054000,33.837280,22.344000,13815.170426
std,13158.304093,65.845703,69.718656,1833.574276
min,536370.000000,0.290000,1.000000,12388.000000
25%,547043.500000,8.500000,3.000000,12553.000000
50%,559307.500000,15.600000,9.000000,12683.000000
75%,569900.000000,27.200000,18.000000,15124.000000
max,581492.000000,633.600000,1152.000000,18287.000000


## 3. Perguntas da análise

### Qual é o valor total vendido?

In [5]:
round(df['OrderValue'].sum(), 2)

np.float64(16918.64)

### Qual país vende mais?

In [6]:
by_country = total_by_country(df)
by_country

,Country,OrderValue
0,Australia,6349.89
1,Germany,3153.71
2,United Kingdom,3131.01
3,France,2623.79
4,Hong Kong,1660.24


### Quantos pedidos e clientes há por país?

In [7]:
df.groupby('Country').agg(
    pedidos=('InvoiceNo', 'nunique'),
    clientes=('CustomerID', 'nunique'),
    itens=('Quantity', 'sum'),
).sort_values('pedidos', ascending=False)

,pedidos,clientes,itens
Country,,,
United Kingdom,195,140,1483
Germany,83,49,1951
France,81,43,1587
Australia,28,6,5325
Hong Kong,7,0,826


## 4. Salvar o resumo
Os resultados tratados vão para `data/processed/`.

In [8]:
save_processed(by_country, 'total_by_country')

PosixPath('/home/cmsouza/PycharmProjects/Plotly/data/processed/total_by_country.csv')

## Conclusões

- 500 registros de vendas em **5 países**, somando **16.918,64** em pedidos.
- **Australia** lidera com **6.349,89**, quase o dobro do 2º lugar (Germany, 3.153,71).
- **Hong Kong** tem o menor total (**1.660,24**).
- Não há linhas duplicadas; a coluna `CustomerID` tem **101 nulos** (vendas sem cliente identificado), o que não afeta os totais por país.